# GRU

In [1]:
#安裝匯入套件
# ! pip install seaborn
! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/work/claire901114/NLP_HW2' 

In [2]:
# 訓練與驗證集
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
df_train

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177
...,...,...
2369245,1+(37*8)=,297
2369246,37-25-19=,-7
2369247,7+39-40=,6
2369248,27-28-12=,-13


In [4]:
df_eval

,src,tgt
0,48+43+34=,125
1,30-(48+13)=,-31
2,(21*31)+10=,661
3,2-27-10=,-35
4,(15*20)+24=,324
...,...,...
263245,14*43*23=,13846
263246,48-(5*27)=,-87
263247,30*42+16=,1276
263248,21*(10-15)=,-105


In [3]:
# 將數據轉換為字串格式
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

In [4]:
# === 構建字元映射表 (Dictionary Building) ===
char_to_id = {}
id_to_char = {}

# 加特殊符號
char_to_id['<pad>'] = 0  # 用於填充序列
char_to_id['<eos>'] = 1  # 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集資料集中的唯一字元
# 彙整訓練資料中出現過的所有字元，為建立詞彙表做準備。
unique_chars = set(''.join(df_train['src']))
# 移除已手動定義的特殊符號，避免重複分配 ID
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# 穩定性優化：排序字元
# 透過排序確保每次執行程式時，字元與 ID 的對應關係保持一致，避免模型訓練的隨機性。
sorted_chars = sorted(list(unique_chars))

# 每個字符分配ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))


Vocab size: 18


In [5]:
# === 資料預處理：構建訓練序列與遮罩標籤 ===
# 將原始資料轉換為模型所需的輸入與輸出格式，並加入序列結束符號 <eos>。
def build_masked_target_ids(text, mapping, pad_id):
    """
    生成輸入 ID 序列與帶遮罩的目標 ID 序列。
    模型應專注於預測「等號後」的答案，而非重複輸入的算式。
    """
    # 1. 完整輸入 ID 序列 (含 <eos>)
    input_ids = [mapping.get(ch, pad_id) for ch in text] + [mapping['<eos>']]

    # 2. 標準的目標 ID 序列 (往左平移)
    # Target: I_1, I_2, ..., I_N, PAD
    target_ids = input_ids[1:] + [pad_id]

    # 3. 實作遮罩邏輯 (等號前的預測不計入loss(設為IGNORE_INDEX)，僅計算答案部分的 Loss)
    masked_target_ids = []

    # 找到等號的位置，並處理等號前的序列
    found_equal = False
    for i, ch in enumerate(text):
        if ch == '=':
            found_equal = True

        # 判斷當前 token 是否是答案的一部分（即等號後的第一個數字或 <eos>）
        # 目標序列的第 i 個元素對應的是輸入序列的第 i+1 個元素。

        if not found_equal:
            # 等號前，目標 ID 設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)
        else:
            # 等號之後，目標 ID 設為正常值
            masked_target_ids.append(target_ids[i])

    # 4. 處理 <eos> 的目標 ID
    # 序列的最後一個目標是 PAD_ID (target_ids 的最後一個元素)，它對應的是 <eos> 的輸入。
    # 確保 masked_target_ids 的長度與 target_ids 一致

    # 從 target_ids 的角度進行遮罩
    masked_target_ids = []

    # 標記等號在 input_ids中的位置
    equal_idx = -1
    try:
        equal_idx = input_ids.index(char_to_id['='])
    except ValueError:
        # 若算式中未包含等號（異常資料）
        pass

    for i in range(len(target_ids)):
        # 判定邏輯：i 對應的是模型預測 Input[i] 後產出的 Target[i]
        # 僅當目標位置處於等號之後（即預測結果為答案的一部分），才保留真實標籤
        if equal_idx != -1 and i >= equal_idx:
             # 如果目標是對應答案的 ID (在 '=' 之後)
            masked_target_ids.append(target_ids[i])
        else:
            # 等號前或等號本身的目標，設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)

    return input_ids, masked_target_ids


# 處理訓練集：生成特徵 (char_id_list)、標籤 (label_id_list) 並統計序列長度
results_train = df_train['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_train['char_id_list'] = [r[0] for r in results_train]
df_train['label_id_list'] = [r[1] for r in results_train]
df_train['len'] = df_train['char_id_list'].apply(len)

# 處理驗證集：確保驗證邏輯與訓練一致
results_eval = df_eval['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_eval['char_id_list'] = [r[0] for r in results_eval]
df_eval['label_id_list'] = [r[1] for r in results_eval]
df_eval['len'] = df_eval['char_id_list'].apply(len)

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=882,882,15,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 15, 15, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 15, 9, 1, 0]"
1,(6+1)*5=35,35,11,"[2, 13, 5, 8, 3, 4, 12, 17, 10, 12, 1]","[0, 0, 0, 0, 0, 0, 0, 10, 12, 1, 0]"
2,13+32+29=74,74,12,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 14, 11, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 1, 0]"
3,31*(3-11)=-248,-248,15,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1, 0]"
4,24*49+1=1177,1177,13,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 14, 14, 1]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 14, 14, 1, 0]"


In [9]:
# 超參數設定
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001 
grad_clip = 1

In [10]:
# === 資料批次處理 (Data Batching) ===
# 運用 PyTorch 標準封裝，將預處理好的 ID 序列轉換為模型可高效讀取的 Batch 格式。

class Dataset(torch.utils.data.Dataset):
    """
    將 Pandas DataFrame 封裝為 PyTorch Dataset。
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # 回傳總資料筆數，讓 DataLoader 知道取樣範圍
        return len(self.sequences)

    def __getitem__(self, index):
        # 根據給定的索引 (index) 從資料集中提取一組訓練樣本
        # x: 輸入字元 ID 列表 (char_id_list)
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here
        # y: 帶有損失遮罩的目標標籤 ID 列表 (label_id_list)
        y = self.sequences.iloc[index]['label_id_list'] # Write your code here
        return x, y

# collate function：用於動態處理 Batch 內的序列對齊
def collate_fn(batch):
    # 轉換為 Tensor 格式，為後續運算做準備
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]

    # 記錄每個樣本的原始長度
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Padding：每個 Batch中的句子長度不同， pad_sequence 補齊至該 Batch 的最大長度，才能以Tensor進行平行計算
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [11]:
# 封裝訓練集資料
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [12]:
# 建立數據加載器：負責打亂資料順序與 Batching
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train, 
    batch_size=batch_size,
    shuffle=True,       # 打亂資料順序 增強泛化能力
    collate_fn=collate_fn
)

In [13]:
# === GRU Model Design ===
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        
        # Embedding Layer
        # 將字元 ID 轉換為稠密向量，設定 padding_idx 以自動忽略填充符號
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                              embedding_dim=embed_dim,
                                              padding_idx=char_to_id['<pad>'])
        
        # GRU Layers
        self.rnn_layer1 = torch.nn.GRU(input_size=embed_dim,      
                                         hidden_size=hidden_dim,
                                         batch_first=True)
        
        self.rnn_layer2 = torch.nn.GRU(input_size=hidden_dim,     
                                         hidden_size=hidden_dim,
                                         batch_first=True)
        # Fully Connected Layer
        # 透過多層感知器 (MLP) 將隱含狀態映射回詞彙表維度，進行字元分類
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                              out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                              out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    def encoder(self, batch_x, batch_x_lens):
        # 向量化處理
        batch_x = self.embedding(batch_x)
        # 效能優化：封裝變長序列
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        # 特徵提取
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        # 還原序列
        # 將 Pack 格式轉回 Tensor，以便進行後續的 Linear 層運算
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        # 輸出預測機率
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
           """
            推理函式：給定起始算式，由模型自主預測後續答案。
           """
            char_list = [char_to_id[c] for c in start_char]
        
            next_char = None

            # 評估模式
            self.eval()

            # 使用 torch.no_grad() 上下文管理器
            with torch.no_grad():
                while len(char_list) < max_len: 
                    
                    # 準備模型的輸入張量
                    device = next(self.parameters()).device
                    input_tensor = torch.tensor([char_list], dtype=torch.long).to(device)
                
                    # 列的實際長度
                    input_length = torch.tensor([len(char_list)], dtype=torch.long)
                
                    # 將輸入傳入模型以獲得預測的 logits
                    # 輸出 y 維度: [批次大小, 序列長度, 字典大小]
                    #             [1, len(char_list), vocab_size]
                    y = self.forward(input_tensor, input_length)
                
                    # 只關心對「下一個」字元的預測，這對應於序列中「最後一個」時間點的輸出
                    # 使用 y[:, -1, :] 來選取這個部分，得到維度 [1, vocab_size]
                    last_time_step_logits = y[:, -1, :]
                
                    #使用 argmax 找出分數最高的那個字元的 ID
                    #torch.argmax 回傳的是張量，所以使用 .item()提取數字
                    next_char = torch.argmax(last_time_step_logits, dim=1).item()

                    # 檢查生成的字元是否為序列結束符號
                    if next_char == char_to_id['<eos>']:
                        break
                
                    # 如果不是，將新字元的 ID 加入到我們的列表中，並繼續迴圈
                    char_list.append(next_char)
            
            # 將最終的 ID 列表轉換回字元，並回傳結果
            return [id_to_char[ch_id] for ch_id in char_list]

In [14]:
torch.manual_seed(2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim).to(device) 

In [15]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# 使用 Adam 優化器
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) 

In [17]:
# Training
from tqdm import tqdm
from copy import deepcopy

# 設為訓練模式 
model = model.to(device)
model.train()

# i 控制何時印出 loss
i = 0 
for epoch in range(1, epochs+1):
    # --- 訓練迴圈 ---
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        
        # 清除梯度
        # 在計算新的梯度前，必須先清除上一步遺留的梯度
        optimizer.zero_grad()
    
        # 將資料移動到 GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 模型前向傳播，得到預測結果
        # batch_pred_y 的維度: [batch_size, seq_len, vocab_size]
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # 計算損失、反向傳播 
        # CrossEntropyLoss 要求 pred 維度為 (N, C) 和 target 維度為 (N)，需要將 batch 和 seq_len 攤平
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = batch_y.view(-1)
        
        loss = criterion(pred_view, target_view)
        
        # 損失計算梯度
        loss.backward()

        # gradient clipping，防止梯度爆炸
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) 

        # 更新模型參數 
        # 優化器根據計算出的梯度，更新模型的權重
        optimizer.step()

        i += 1
        if i % 50 == 0:
            bar.set_postfix(loss=loss.item())
    
    # 切換到評估模式
    model.eval()
    
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        # 取出問題和標準答案
        # batch_x 是字串，例如 '10+4='
        # batch_y 也是字串，例如 '14'
        batch_x = row['src']# .replace(row['tgt'], '') # 確保只傳入問題部分
        batch_y = str(row['tgt'])
        
        # 使用 generator 生成預測 
        # generator 會回傳一個字元列表，如 ['1', '0', '+', '4', '=', '1', '4']
        prediction = model.generator(batch_x)
        
        # 從生成結果中提取答案部分
        # 我們的答案是從 '=' 後面開始的
        try:
            equal_idx = prediction.index('=')
            predicted_answer = "".join(prediction[equal_idx+1:])
        except ValueError:
            # 如果模型連 '=' 都沒生成，那一定錯
            predicted_answer = ""
            
        # 比較預測與標準答案 
        # 檢查生成的答案字串是否與標準答案完全一樣
        if predicted_answer == batch_y:
            matched += 1
        
        total += 1
        
        # 更新進度條顯示目前的準確率
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # 切回訓練模式，做下一個 epoch
    model.train()
        
    print(f"\nEpoch {epoch} Validation Accuracy: {matched/total:.4f}")

Train epoch 1: 100%|██████████| 37020/37020 [04:49<00:00, 127.69it/s, loss=0.459]
Validation epoch 1: 263250it [45:27, 96.53it/s, accuracy=0.5336] 



Epoch 1 Validation Accuracy: 0.5336


Train epoch 2: 100%|██████████| 37020/37020 [04:47<00:00, 128.78it/s, loss=0.295]
Validation epoch 2: 263250it [41:57, 104.56it/s, accuracy=0.5736]



Epoch 2 Validation Accuracy: 0.5736


Train epoch 3: 100%|██████████| 37020/37020 [04:47<00:00, 128.63it/s, loss=0.297]
Validation epoch 3: 263250it [42:22, 103.54it/s, accuracy=0.6016]


...


Epoch 8 Validation Accuracy: 0.6864


Train epoch 9: 100%|██████████| 37020/37020 [04:42<00:00, 131.26it/s, loss=0.145]
Validation epoch 9: 263250it [40:07, 109.37it/s, accuracy=0.7260]



Epoch 9 Validation Accuracy: 0.7260


Train epoch 10: 100%|██████████| 37020/37020 [04:39<00:00, 132.30it/s, loss=0.0965]
Validation epoch 10: 263250it [39:55, 109.88it/s, accuracy=0.7366]


Epoch 10 Validation Accuracy: 0.7366


# B.	Distribution Shift：訓練含三位數、驗證只含兩位數

In [32]:
# 生成三位數資料集
import random
import pandas as pd
from tqdm import tqdm

NUM_SAMPLES = 2369250  # 樣本數設定
OPERATORS = ['+', '-', '*']

# 加權樣板（讓括號運算比例合理）
TEMPLATES = [
    ("{N1}{op1}{N2}", 0.6),
    ("{N1}{op1}{N2}{op2}{N3}", 0.2),
    ("({N1}{op1}{N2}){op2}{N3}", 0.1),
    ("{N1}{op1}({N2}{op2}{N3})", 0.1),
]

random.seed(42)

def generate_arithmetic_data(num_samples, min_num, max_num, label):
    """
    生成算術題資料。
    label：用來標記「2digit」或「3digit」
    """
    data = []
    print(f"Generating {num_samples:,} {label} samples (range {min_num}-{max_num})...")

    for _ in tqdm(range(num_samples), desc=f"Generating {label} samples"):
        # 隨機選擇
        templates, weights = zip(*TEMPLATES)
        template = random.choices(templates, weights=weights, k=1)[0]

        # 隨機生成數字與運算符
        n1 = random.randint(min_num, max_num)
        n2 = random.randint(min_num, max_num)
        n3 = random.randint(min_num, max_num)
        op1 = random.choice(OPERATORS)
        op2 = random.choice(OPERATORS)

        expr = template.format(N1=n1, N2=n2, N3=n3, op1=op1, op2=op2)
        src = f"{expr}="

        try:
            tgt = str(eval(expr))
            # 排除無效值與過長結果
            if "e" in tgt or "inf" in tgt or len(tgt) > 8:
                continue
            data.append({'src': src, 'tgt': tgt, 'type': label})
        except Exception:
            continue

    # 重複的刪除
    df = pd.DataFrame(data).drop_duplicates(subset=["src"]).reset_index(drop=True)
    return df


# =====================================================
# 混合 2 位 + 3 位數資料
# =====================================================
if __name__ == "__main__":
    # 40% 兩位數、60% 三位數
    ratio_2digit = 0.4
    n_2digit = int(NUM_SAMPLES * ratio_2digit)
    n_3digit = NUM_SAMPLES - n_2digit

    # 生成資料
    df_2digit = generate_arithmetic_data(n_2digit, 10, 99, "2digit")
    df_3digit = generate_arithmetic_data(n_3digit, 100, 999, "3digit")

    # 合併資料
    df_all = pd.concat([df_2digit, df_3digit], ignore_index=True)

    # 若因為重複或過濾導致數量不足，隨機補樣
    total_now = len(df_all)
    if total_now < NUM_SAMPLES:
        diff = NUM_SAMPLES - total_now
        print(f"⚠️ 現有 {total_now:,} 筆，不足 {diff:,}，隨機補樣中...")
        extra = generate_arithmetic_data(diff, 10, 999, "extra")
        df_all = pd.concat([df_all, extra], ignore_index=True)

    # 打亂順序
    df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

    # 儲存
    output_filename = "/work/claire901114/NLP_HW2/arithmetic_mixed_2_3digit.csv"
    df_all.to_csv(output_filename, index=False)

    # 結果
    total = len(df_all)
    count_2 = (df_all["type"] == "2digit").sum()
    count_3 = (df_all["type"] == "3digit").sum()
    print(f"\n✅ Successfully generated {total:,} samples (fixed total = {NUM_SAMPLES:,})")
    print(f"  • 2-digit samples: {count_2:,} ({count_2/total*100:.1f}%)")
    print(f"  • 3-digit samples: {count_3:,} ({count_3/total*100:.1f}%)")
    print("\nData preview:")
    print(df_all.sample(5, random_state=1))


Generating 947,700 2digit samples (range 10-99)...


Generating 2digit samples: 100%|██████████| 947700/947700 [00:12<00:00, 78703.92it/s]


Generating 1,421,550 3digit samples (range 100-999)...


Generating 3digit samples: 100%|██████████| 1421550/1421550 [00:18<00:00, 78059.26it/s]


⚠️ 現有 1,652,807 筆，不足 716,443，隨機補樣中...
Generating 716,443 extra samples (range 10-999)...


Generating extra samples: 100%|██████████| 716443/716443 [00:09<00:00, 78394.59it/s]



✅ Successfully generated 2,325,839 samples (fixed total = 2,369,250)
  • 2-digit samples: 399,491 (17.2%)
  • 3-digit samples: 1,253,316 (53.9%)

Data preview:
                    src      tgt    type
1589199    947-545*501=  -272098  3digit
2049270     (88+94)+92=      274  2digit
2098973    877*421+610=   369827  3digit
1668377       63*44+56=     2828  2digit
996674   (161+415)+334=      910   extra


In [2]:
import pandas as pd
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_mixed_2_3digit.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))


In [3]:
df_train

,src,tgt,type
0,(42*12)+40=,544,2digit
1,475+753=,1228,extra
2,(650*365)-460=,236790,3digit
3,724+92-269=,547,extra
4,35-(94*84)=,-7861,2digit
...,...,...,...
2325834,280*109=,30520,3digit
2325835,(43-15)+55=,83,2digit
2325836,763*101=,77063,extra
2325837,711-281=,430,extra


In [4]:
# 將數據轉換為字串格式
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

In [5]:
# === 構建字元映射表 (Dictionary Building) ===
char_to_id = {}
id_to_char = {}

# 加特殊符號
char_to_id['<pad>'] = 0  # 用於填充序列
char_to_id['<eos>'] = 1  # 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集資料集中的唯一字元
# 彙整訓練資料中出現過的所有字元，為建立詞彙表做準備。
unique_chars = set(''.join(df_train['src']))
# 移除已手動定義的特殊符號，避免重複分配 ID
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# 穩定性優化：排序字元
# 透過排序確保每次執行程式時，字元與 ID 的對應關係保持一致，避免模型訓練的隨機性。
sorted_chars = sorted(list(unique_chars))

# 每個字符分配ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))



Vocab size: 18


In [6]:
# === 資料預處理：構建訓練序列與遮罩標籤 ===
# 將原始資料轉換為模型所需的輸入與輸出格式，並加入序列結束符號 <eos>。
def build_masked_target_ids(text, mapping, pad_id):
    """
    生成輸入 ID 序列與帶遮罩的目標 ID 序列。
    模型應專注於預測「等號後」的答案，而非重複輸入的算式。
    """
    # 1. 完整輸入 ID 序列 (含 <eos>)
    input_ids = [mapping.get(ch, pad_id) for ch in text] + [mapping['<eos>']]

    # 2. 標準的目標 ID 序列 (往左平移)
    # Target: I_1, I_2, ..., I_N, PAD
    target_ids = input_ids[1:] + [pad_id]

    # 3. 實作遮罩邏輯 (等號前的預測不計入loss(設為IGNORE_INDEX)，僅計算答案部分的 Loss)
    masked_target_ids = []

    # 找到等號的位置，並處理等號前的序列
    found_equal = False
    for i, ch in enumerate(text):
        if ch == '=':
            found_equal = True

        # 判斷當前 token 是否是答案的一部分（即等號後的第一個數字或 <eos>）
        # 目標序列的第 i 個元素對應的是輸入序列的第 i+1 個元素。

        if not found_equal:
            # 等號前，目標 ID 設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)
        else:
            # 等號之後，目標 ID 設為正常值
            masked_target_ids.append(target_ids[i])

    # 4. 處理 <eos> 的目標 ID
    # 序列的最後一個目標是 PAD_ID (target_ids 的最後一個元素)，它對應的是 <eos> 的輸入。
    # 確保 masked_target_ids 的長度與 target_ids 一致

    # 從 target_ids 的角度進行遮罩
    masked_target_ids = []

    # 標記等號在 input_ids中的位置
    equal_idx = -1
    try:
        equal_idx = input_ids.index(char_to_id['='])
    except ValueError:
        # 若算式中未包含等號（異常資料）
        pass

    for i in range(len(target_ids)):
        # 判定邏輯：i 對應的是模型預測 Input[i] 後產出的 Target[i]
        # 僅當目標位置處於等號之後（即預測結果為答案的一部分），才保留真實標籤
        if equal_idx != -1 and i >= equal_idx:
             # 如果目標是對應答案的 ID (在 '=' 之後)
            masked_target_ids.append(target_ids[i])
        else:
            # 等號前或等號本身的目標，設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)

    return input_ids, masked_target_ids


# 處理訓練集：生成特徵 (char_id_list)、標籤 (label_id_list) 並統計序列長度
results_train = df_train['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_train['char_id_list'] = [r[0] for r in results_train]
df_train['label_id_list'] = [r[1] for r in results_train]
df_train['len'] = df_train['char_id_list'].apply(len)

# 處理驗證集：確保驗證邏輯與訓練一致
results_eval = df_eval['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_eval['char_id_list'] = [r[0] for r in results_eval]
df_eval['label_id_list'] = [r[1] for r in results_eval]
df_eval['len'] = df_eval['char_id_list'].apply(len)

df_train.head()

,src,tgt,type,len,char_id_list,label_id_list
0,(42*12)+40=544,544,2digit,15,"[2, 11, 9, 4, 8, 9, 3, 5, 11, 7, 17, 12, 11, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 11, 11, 1, 0]"
1,475+753=1228,1228,extra,13,"[11, 14, 12, 5, 14, 12, 10, 17, 8, 9, 9, 15, 1]","[0, 0, 0, 0, 0, 0, 0, 8, 9, 9, 15, 1, 0]"
2,(650*365)-460=236790,236790,3digit,21,"[2, 13, 12, 7, 4, 10, 13, 12, 3, 6, 11, 13, 7,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 10,..."
3,724+92-269=547,547,extra,15,"[14, 9, 11, 5, 16, 9, 6, 9, 13, 16, 17, 12, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 11, 14, 1, 0]"
4,35-(94*84)=-7861,-7861,2digit,17,"[10, 12, 6, 2, 16, 11, 4, 15, 11, 3, 17, 6, 14...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 14, 15, 13, ..."


In [7]:
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001 
grad_clip = 1

In [8]:
# === 資料批次處理 (Data Batching) ===
# 運用 PyTorch 標準封裝，將預處理好的 ID 序列轉換為模型可高效讀取的 Batch 格式。

class Dataset(torch.utils.data.Dataset):
    """
    將 Pandas DataFrame 封裝為 PyTorch Dataset。
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # 回傳總資料筆數，讓 DataLoader 知道取樣範圍
        return len(self.sequences)

    def __getitem__(self, index):
        # 根據給定的索引 (index) 從資料集中提取一組訓練樣本
        # x: 輸入字元 ID 列表 (char_id_list)
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here
        # y: 帶有損失遮罩的目標標籤 ID 列表 (label_id_list)
        y = self.sequences.iloc[index]['label_id_list'] # Write your code here
        return x, y

# collate function：用於動態處理 Batch 內的序列對齊
def collate_fn(batch):
    # 轉換為 Tensor 格式，為後續運算做準備
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]

    # 記錄每個樣本的原始長度
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Padding：每個 Batch中的句子長度不同， pad_sequence 補齊至該 Batch 的最大長度，才能以Tensor進行平行計算
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens
    
    
    
# 封裝訓練集資料
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])


# 建立數據加載器：負責打亂資料順序與 Batching
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train, 
    batch_size=batch_size,
    shuffle=True,       # 打亂資料順序 增強泛化能力
    collate_fn=collate_fn
)


In [11]:
ds_eval = Dataset(df_eval[['char_id_list', 'label_id_list']])

dl_eval = torch.utils.data.DataLoader(
    dataset=ds_eval,
    batch_size=batch_size,
    shuffle=False,       
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [12]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        # Embedding Layer
        # 將離散的字元 ID 映射為連續的向量空間，並設定 padding_idx 確保填充標記不參與梯度更新
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])

        # LSTM Layers：使用雙層 LSTM 結構以捕捉資料中更深層的序列依賴關係
        # batch_first=True 確保輸入張量維度為 [Batch, Seq, Feature]
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        # Fully Connected Layer 
        # 透過線性變換與 ReLU 激活函數，將 LSTM 的隱藏狀態映射回詞彙表維度
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        """前向傳播"""
        return self.encoder(batch_x, batch_x_lens)
    
    def encoder(self, batch_x, batch_x_lens):
        """模型編碼邏輯，將字元序列轉化為預測機率"""
        # 向量化特徵提取
        batch_x = self.embedding(batch_x)
        # 針對變長序列進行優化，略過填充部分，提升計算效率
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        # D. 序列還原：將壓縮格式還原為標準張量，以便進入全連接層
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        # 映射至字元機率分佈
        batch_x = self.linear(batch_x)
        return batch_x
    
    def generator(self, start_char, max_len=200):
           """
           給定起始字串，採自迴歸方式預測後續字元
           """
            # 將起始字元轉為 ID 列表
            char_list = [char_to_id[c] for c in start_char]
        
            next_char = None

            # 評估模式
            self.eval()

            # 使用torch.no_grad()上下文管理
            with torch.no_grad():
                while len(char_list) < max_len: 

                    # 1. 準備模型的輸入張量
                    device = next(self.parameters()).device
                    #    輸入需要有批次維度，所以我們將 char_list 包在另一個列表中
                    #    維度變為: [1, 當前序列長度]
                    input_tensor = torch.tensor([char_list], dtype=torch.long).to(device)
                
                    # 模型的 forward 方法也需要序列的實際長度
                    input_length = torch.tensor([len(char_list)], dtype=torch.long)
                
                    # 2. 將輸入傳入模型以獲得預測的logits
                    y = self.forward(input_tensor, input_length)
                
                    # 3. 我們只關心對「下一個」字元的預測，這對應於序列中「最後一個」時間點的輸出
                    last_time_step_logits = y[:, -1, :]
                
                    # 4. 使用argmax找出分數最高的字元的id
                    next_char = torch.argmax(last_time_step_logits, dim=1).item()
                

                    # 5. 檢查生成的字元是否為序列結束符號
                    if next_char == char_to_id['<eos>']:
                        break
                
                    # 6. 如果不是，將新字元的id加入到我們的列表中，並繼續迴圈
                    char_list.append(next_char)
            
            # 最終的 ID 列表轉換回字元，並回傳結果
            return [id_to_char[ch_id] for ch_id in char_list]

In [13]:
torch.manual_seed(2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim).to(device) # 將模型移動到指定的設備上

In [14]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# 使用 Adam 優化器
# optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) 

In [18]:
from tqdm import tqdm
from copy import deepcopy

# 設為訓練模式 
model = model.to(device)
model.train()

# i 控制何時印出 loss
i = 0 
for epoch in range(1, epochs+1):
    # 訓練迴圈
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        
        # 清除梯度:在計算新的梯度前，必須先清除上一步遺留的梯度
        optimizer.zero_grad()
    
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 模型前向傳播，得到預測結果
        # batch_pred_y 的維度: [batch_size, seq_len, vocab_size]
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # 計算損失 & 反向傳播:CrossEntropyLoss要求 pred 維度為 (N, C) 和 target 維度為 (N)，所以要將batch維度和seq_len維度攤平
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = batch_y.view(-1)
        
        loss = criterion(pred_view, target_view)
        
        # 損失計算梯度
        loss.backward()

        # gradient clipping防止梯度爆炸
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) 

        #更新模型參數：優化器根據計算出的梯度來更新模型的權重
        optimizer.step()

        i += 1
        if i % 50 == 0:
            bar.set_postfix(loss=loss.item())
    
    # 切到評估模式 
    model.eval()
    
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        # 從 DataFrame 中取出問題和標準答案
        batch_x = row['src']
        batch_y = str(row['tgt'])
        
        # 使用 generator 生成預測 
        prediction = model.generator(batch_x)
        
        # 答案是從 = 後面開始的
        try:
            equal_idx = prediction.index('=')
            predicted_answer = "".join(prediction[equal_idx+1:])
        except ValueError:
            predicted_answer = ""
            
        # 比較預測與標準答案
        if predicted_answer == batch_y:
            matched += 1
        
        total += 1
        
        # 更新進度條
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # 切回訓練，下一個 epoch
    model.train()
        
    print(f"\nEpoch {epoch} Validation Accuracy: {matched/total:.4f}")

Train epoch 1: 100%|██████████| 36342/36342 [03:21<00:00, 180.22it/s, loss=0.596]
Validation epoch 1: 263250it [40:25, 108.52it/s, accuracy=0.2947]



Epoch 1 Validation Accuracy: 0.2947


Train epoch 2: 100%|██████████| 36342/36342 [03:21<00:00, 180.09it/s, loss=0.518]
Validation epoch 2: 263250it [40:48, 107.52it/s, accuracy=0.3155]



Epoch 2 Validation Accuracy: 0.3155


Train epoch 3: 100%|██████████| 36342/36342 [03:21<00:00, 180.54it/s, loss=0.446]
Validation epoch 3: 263250it [40:32, 108.23it/s, accuracy=0.3154]


...


Epoch 8 Validation Accuracy: 0.3604


Train epoch 9: 100%|██████████| 36342/36342 [03:25<00:00, 177.26it/s, loss=0.462]
Validation epoch 9: 263250it [41:25, 105.89it/s, accuracy=0.3600]



Epoch 9 Validation Accuracy: 0.3600


Train epoch 10: 100%|██████████| 36342/36342 [03:21<00:00, 180.22it/s, loss=0.427]
Validation epoch 10: 263250it [42:08, 104.13it/s, accuracy=0.3615]



Epoch 10 Validation Accuracy: 0.3615


# C.	訓練資料含 20% 錯誤答案（label noise）

In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [6]:
import random
#--------------------------------------------------------
# 建立生成20%錯誤答案
#--------------------------------------------------------
n_corrupt = int(len(df_train) * 0.2)
idx_to_corrupt = random.sample(range(len(df_train)), n_corrupt)

def random_wrong_answer(true_answer):
    """回傳與原答案不同的隨機整數"""
    try:
        ans = int(true_answer)
        wrong = ans
        while wrong == ans:
            wrong = random.randint(0, 99)
        return wrong
    except:
        return -1  # 若有非整數資料，預設為 -1

# 確保欄位型別為 int
df_train["tgt"] = pd.to_numeric(df_train["tgt"], errors="coerce").fillna(-1).astype(int)

# 改 20% 答案
for i in idx_to_corrupt:
    df_train.loc[i, "tgt"] = random_wrong_answer(df_train.loc[i, "tgt"])


# 確認
print("\nAfter corruption (sample):")
print(df_train.head(10))
print("\ndf_train.dtypes:\n", df_train.dtypes)


After corruption (sample):
           src   tgt
0  14*(43+20)=    12
1     (6+1)*5=    91
2    13+32+29=    92
3   31*(3-11)=  -248
4     24*49+1=  1177
5   3+(25*25)=   628
6   8*(30+10)=   320
7     9*38+49=   391
8       23-17=     6
9    23-26*15=  -367

df_train.dtypes:
 src    object
tgt     int64
dtype: object


In [7]:
# 儲存新資料集
output_path = os.path.join(data_path, 'arithmetic_train_20wrong.csv')
df_train.to_csv(output_path, index=False)
print(f"\n✅ Saved corrupted file as: {output_path}")


✅ Saved corrupted file as: /work/claire901114/NLP_HW2/arithmetic_train_20wrong.csv


In [9]:
# 把已生成20%錯誤答案去訓練
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train_20wrong.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train

,src,tgt
0,14*(43+20)=,12
1,(6+1)*5=,91
2,13+32+29=,92
3,31*(3-11)=,-248
4,24*49+1=,1177
...,...,...
2369245,1+(37*8)=,297
2369246,37-25-19=,92
2369247,7+39-40=,6
2369248,27-28-12=,-13


In [10]:
# 將數據轉換為字串格式
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

In [11]:
# === 構建字元映射表 (Dictionary Building) ===
char_to_id = {}
id_to_char = {}

# 加特殊符號
char_to_id['<pad>'] = 0  # 用於填充序列
char_to_id['<eos>'] = 1  # 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集資料集中的唯一字元
# 彙整訓練資料中出現過的所有字元，為建立詞彙表做準備。
unique_chars = set(''.join(df_train['src']))
# 移除已手動定義的特殊符號，避免重複分配 ID
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# 穩定性優化：排序字元
# 透過排序確保每次執行程式時，字元與 ID 的對應關係保持一致，避免模型訓練的隨機性。
sorted_chars = sorted(list(unique_chars))

# 每個字符分配ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))

Vocab size: 18


In [12]:
# === 資料預處理：構建訓練序列與遮罩標籤 ===
# 將原始資料轉換為模型所需的輸入與輸出格式，並加入序列結束符號 <eos>。
def build_masked_target_ids(text, mapping, pad_id):
    """
    生成輸入 ID 序列與帶遮罩的目標 ID 序列。
    模型應專注於預測「等號後」的答案，而非重複輸入的算式。
    """
    # 1. 完整輸入 ID 序列 (含 <eos>)
    input_ids = [mapping.get(ch, pad_id) for ch in text] + [mapping['<eos>']]

    # 2. 標準的目標 ID 序列 (往左平移)
    # Target: I_1, I_2, ..., I_N, PAD
    target_ids = input_ids[1:] + [pad_id]

    # 3. 實作遮罩邏輯 (等號前的預測不計入loss(設為IGNORE_INDEX)，僅計算答案部分的 Loss)
    masked_target_ids = []

    # 找到等號的位置，並處理等號前的序列
    found_equal = False
    for i, ch in enumerate(text):
        if ch == '=':
            found_equal = True

        # 判斷當前 token 是否是答案的一部分（即等號後的第一個數字或 <eos>）
        # 目標序列的第 i 個元素對應的是輸入序列的第 i+1 個元素。

        if not found_equal:
            # 等號前，目標 ID 設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)
        else:
            # 等號之後，目標 ID 設為正常值
            masked_target_ids.append(target_ids[i])

    # 4. 處理 <eos> 的目標 ID
    # 序列的最後一個目標是 PAD_ID (target_ids 的最後一個元素)，它對應的是 <eos> 的輸入。
    # 確保 masked_target_ids 的長度與 target_ids 一致

    # 從 target_ids 的角度進行遮罩
    masked_target_ids = []

    # 標記等號在 input_ids中的位置
    equal_idx = -1
    try:
        equal_idx = input_ids.index(char_to_id['='])
    except ValueError:
        # 若算式中未包含等號（異常資料）
        pass

    for i in range(len(target_ids)):
        # 判定邏輯：i 對應的是模型預測 Input[i] 後產出的 Target[i]
        # 僅當目標位置處於等號之後（即預測結果為答案的一部分），才保留真實標籤
        if equal_idx != -1 and i >= equal_idx:
             # 如果目標是對應答案的 ID (在 '=' 之後)
            masked_target_ids.append(target_ids[i])
        else:
            # 等號前或等號本身的目標，設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)

    return input_ids, masked_target_ids


# 處理訓練集：生成特徵 (char_id_list)、標籤 (label_id_list) 並統計序列長度
results_train = df_train['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_train['char_id_list'] = [r[0] for r in results_train]
df_train['label_id_list'] = [r[1] for r in results_train]
df_train['len'] = df_train['char_id_list'].apply(len)

# 處理驗證集：確保驗證邏輯與訓練一致
results_eval = df_eval['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_eval['char_id_list'] = [r[0] for r in results_eval]
df_eval['label_id_list'] = [r[1] for r in results_eval]
df_eval['len'] = df_eval['char_id_list'].apply(len)

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=12,12,14,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 8, 9, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 9, 1, 0]"
1,(6+1)*5=91,91,11,"[2, 13, 5, 8, 3, 4, 12, 17, 16, 8, 1]","[0, 0, 0, 0, 0, 0, 0, 16, 8, 1, 0]"
2,13+32+29=92,92,12,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 16, 9, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 16, 9, 1, 0]"
3,31*(3-11)=-248,-248,15,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1, 0]"
4,24*49+1=1177,1177,13,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 14, 14, 1]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 14, 14, 1, 0]"


In [13]:
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001 
grad_clip = 1

In [14]:
# === 資料批次處理 (Data Batching) ===
# 運用 PyTorch 標準封裝，將預處理好的 ID 序列轉換為模型可高效讀取的 Batch 格式。

class Dataset(torch.utils.data.Dataset):
    """
    將 Pandas DataFrame 封裝為 PyTorch Dataset。
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # 回傳總資料筆數，讓 DataLoader 知道取樣範圍
        return len(self.sequences)

    def __getitem__(self, index):
        # 根據給定的索引 (index) 從資料集中提取一組訓練樣本
        # x: 輸入字元 ID 列表 (char_id_list)
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here
        # y: 帶有損失遮罩的目標標籤 ID 列表 (label_id_list)
        y = self.sequences.iloc[index]['label_id_list'] # Write your code here
        return x, y

# collate function：用於動態處理 Batch 內的序列對齊
def collate_fn(batch):
    # 轉換為 Tensor 格式，為後續運算做準備
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]

    # 記錄每個樣本的原始長度
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Padding：每個 Batch中的句子長度不同， pad_sequence 補齊至該 Batch 的最大長度，才能以Tensor進行平行計算
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x

In [15]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [16]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train,
    batch_size=batch_size,
    shuffle=True,       # 在訓練時打亂資料順序，這點非常重要
    collate_fn=collate_fn
)

In [17]:
# === LSTM Model Design ===
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        # Embedding Layer
        # 將離散的字元 ID 映射為連續的向量空間，並設定 padding_idx 確保填充標記不參與梯度更新
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])

        # LSTM Layers：使用雙層 LSTM 結構以捕捉資料中更深層的序列依賴關係
        # batch_first=True 確保輸入張量維度為 [Batch, Seq, Feature]
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        # Fully Connected Layer 
        # 透過線性變換與 ReLU 激活函數，將 LSTM 的隱藏狀態映射回詞彙表維度
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        """前向傳播"""
        return self.encoder(batch_x, batch_x_lens)
    
    def encoder(self, batch_x, batch_x_lens):
        """模型編碼邏輯，將字元序列轉化為預測機率"""
        # 向量化特徵提取
        batch_x = self.embedding(batch_x)
        # 針對變長序列進行優化，略過填充部分，提升計算效率
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        # D. 序列還原：將壓縮格式還原為標準張量，以便進入全連接層
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        # 映射至字元機率分佈
        batch_x = self.linear(batch_x)
        return batch_x
    
    def generator(self, start_char, max_len=200):
           """
           給定起始字串，採自迴歸方式預測後續字元
           """
            # 將起始字元轉為 ID 列表
            char_list = [char_to_id[c] for c in start_char]
        
            next_char = None

            # 評估模式
            self.eval()

            # 使用torch.no_grad()上下文管理
            with torch.no_grad():
                while len(char_list) < max_len: 

                    # 1. 準備模型的輸入張量
                    device = next(self.parameters()).device
                    #    輸入需要有批次維度，所以我們將 char_list 包在另一個列表中
                    #    維度變為: [1, 當前序列長度]
                    input_tensor = torch.tensor([char_list], dtype=torch.long).to(device)
                
                    # 模型的 forward 方法也需要序列的實際長度
                    input_length = torch.tensor([len(char_list)], dtype=torch.long)
                
                    # 2. 將輸入傳入模型以獲得預測的logits
                    y = self.forward(input_tensor, input_length)
                
                    # 3. 我們只關心對「下一個」字元的預測，這對應於序列中「最後一個」時間點的輸出
                    last_time_step_logits = y[:, -1, :]
                
                    # 4. 使用argmax找出分數最高的字元的id
                    next_char = torch.argmax(last_time_step_logits, dim=1).item()
                

                    # 5. 檢查生成的字元是否為序列結束符號
                    if next_char == char_to_id['<eos>']:
                        break
                
                    # 6. 如果不是，將新字元的id加入到我們的列表中，並繼續迴圈
                    char_list.append(next_char)
            
            # 最終的 ID 列表轉換回字元，並回傳結果
            return [id_to_char[ch_id] for ch_id in char_list]

In [18]:
torch.manual_seed(2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim).to(device) # 將模型移動到指定的設備上

In [19]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# 使用 Adam 優化器
# optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) 

In [21]:
from tqdm import tqdm
from copy import deepcopy

# 設為訓練模式 
model = model.to(device)
model.train()

# i 控制何時印出 loss
i = 0 
for epoch in range(1, epochs+1):
    # 訓練迴圈
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        
        # 清除梯度:在計算新的梯度前，必須先清除上一步遺留的梯度
        optimizer.zero_grad()
    
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 模型前向傳播，得到預測結果
        # batch_pred_y 的維度: [batch_size, seq_len, vocab_size]
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # 計算損失 & 反向傳播:CrossEntropyLoss要求 pred 維度為 (N, C) 和 target 維度為 (N)，所以要將batch維度和seq_len維度攤平
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = batch_y.view(-1)
        
        loss = criterion(pred_view, target_view)
        
        # 損失計算梯度
        loss.backward()

        # gradient clipping防止梯度爆炸
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) 

        #更新模型參數：優化器根據計算出的梯度來更新模型的權重
        optimizer.step()

        i += 1
        if i % 50 == 0:
            bar.set_postfix(loss=loss.item())
    
    # 切到評估模式 
    model.eval()
    
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        # 從 DataFrame 中取出問題和標準答案
        batch_x = row['src']
        batch_y = str(row['tgt'])
        
        # 使用 generator 生成預測 
        prediction = model.generator(batch_x)
        
        # 答案是從 = 後面開始的
        try:
            equal_idx = prediction.index('=')
            predicted_answer = "".join(prediction[equal_idx+1:])
        except ValueError:
            predicted_answer = ""
            
        # 比較預測與標準答案
        if predicted_answer == batch_y:
            matched += 1
        
        total += 1
        
        # 更新進度條
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # 切回訓練，下一個 epoch
    model.train()
        
    print(f"\nEpoch {epoch} Validation Accuracy: {matched/total:.4f}")

Train epoch 1: 100%|██████████| 37020/37020 [05:32<00:00, 111.45it/s, loss=0.997]
Validation epoch 1: 263250it [47:08, 93.07it/s, accuracy=0.3822] 



Epoch 1 Validation Accuracy: 0.3822


Train epoch 2: 100%|██████████| 37020/37020 [05:32<00:00, 111.41it/s, loss=0.954]
Validation epoch 2: 263250it [43:05, 101.82it/s, accuracy=0.4507]



Epoch 2 Validation Accuracy: 0.4507


Train epoch 3: 100%|██████████| 37020/37020 [05:32<00:00, 111.27it/s, loss=0.959]
Validation epoch 3: 263250it [43:58, 99.76it/s, accuracy=0.4952] 


...


Epoch 8 Validation Accuracy: 0.5621


Train epoch 9: 100%|██████████| 37020/37020 [05:26<00:00, 113.41it/s, loss=0.973]
Validation epoch 9: 263250it [43:27, 100.94it/s, accuracy=0.5738]



Epoch 9 Validation Accuracy: 0.5738


Train epoch 10: 100%|██████████| 37020/37020 [05:26<00:00, 113.31it/s, loss=0.806]
Validation epoch 10: 263250it [42:34, 103.03it/s, accuracy=0.5855]



Epoch 10 Validation Accuracy: 0.5855


# D.	是否使用 gradient clipping

In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

In [4]:
char_to_id = {}
id_to_char = {}

# 添加特殊符號
char_to_id['<pad>'] = 0  # <pad> 用於填充序列
char_to_id['<eos>'] = 1  # <eos> 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集訓練資料中的所有唯一字符 (假設 'src' 欄位包含完整的算式和答案)
# 您的原始代碼：unique_chars = set(''.join(df_train['src']) + ''.join(df_train['tgt']))
# 假設 df_train['src'] 已經是完整的算式（如 '1+2=3'），則只需要它
unique_chars = set(''.join(df_train['src']))
# 確保排除特殊符號，因為它們已經被處理
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# **優化步驟：排序字元**，確保每次執行程式時，ID 的分配都是一致的
sorted_chars = sorted(list(unique_chars))

# 為每個字符分配 ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 已經被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 (安全措施，雖然排序前已移除特殊符號)
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  # ID 依次遞增

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))


Vocab size: 18


In [5]:
# TODO2
# 定義一個獨立的函式來生成標準的目標 ID 序列（往左平移），
# 並且將等號前的部分設為 IGNORE_INDEX (即 PAD_ID = 0) 進行遮罩。
def build_masked_target_ids(text, mapping, pad_id):
    # 1. 完整輸入 ID 序列 (含 <eos>)
    input_ids = [mapping.get(ch, pad_id) for ch in text] + [mapping['<eos>']]

    # 2. 標準的目標 ID 序列 (往左平移)
    # Target: I_1, I_2, ..., I_N, PAD
    target_ids = input_ids[1:] + [pad_id]

    # 3. 實作遮罩邏輯 (等號前不計算損失 -> 設為 IGNORE_INDEX)
    masked_target_ids = []

    # 找到等號的位置，並處理等號前的序列
    # 遍歷原始文字，用於找到 '=' 的位置
    found_equal = False
    for i, ch in enumerate(text):
        if ch == '=':
            found_equal = True

        # 判斷當前 token 是否是答案的一部分（即等號後的第一個數字或 <eos>）
        # 目標序列的第 i 個元素對應的是輸入序列的第 i+1 個元素。
        # 因此，當我們在原始文字中找到 '=' 時，表示 target_ids[i] 應該是答案的第一個 ID。

        if not found_equal:
            # 等號前，目標 ID 設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)
        else:
            # 等號之後，目標 ID 設為正常值
            masked_target_ids.append(target_ids[i])

    # 4. 處理 <eos> 的目標 ID
    # 序列的最後一個目標是 PAD_ID (target_ids 的最後一個元素)，它對應的是 <eos> 的輸入。
    # 這裡只需要確保 masked_target_ids 的長度與 target_ids 一致
    # 由於 target_ids 長度是 len(text) + 1，而 masked_target_ids 只有 len(text) 個元素

    # 正確做法：從 target_ids 的角度進行遮罩
    masked_target_ids = []

    # 標記等號在 input_ids 中的位置
    equal_idx = -1
    try:
        equal_idx = input_ids.index(char_to_id['='])
    except ValueError:
        # 如果算式沒有等號，則整個序列都應該是 IGNORE_INDEX
        pass

    for i in range(len(target_ids)):
        # i 是目標序列的索引
        # 目標 ID (target_ids[i]) 對應於輸入序列的 X_{i+1}
        # 如果 i+1 在等號之後（即 i+1 > equal_idx），則該目標才需要計算損失
        if equal_idx != -1 and i >= equal_idx:
             # 如果目標是對應答案的 ID (在 '=' 之後)
            masked_target_ids.append(target_ids[i])
        else:
            # 等號前或等號本身的目標，設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)

    return input_ids, masked_target_ids


# 訓練集
# 假設 PAD_ID = 0
results_train = df_train['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_train['char_id_list'] = [r[0] for r in results_train]
df_train['label_id_list'] = [r[1] for r in results_train]
df_train['len'] = df_train['char_id_list'].apply(len)

# 驗證集
results_eval = df_eval['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_eval['char_id_list'] = [r[0] for r in results_eval]
df_eval['label_id_list'] = [r[1] for r in results_eval]
df_eval['len'] = df_eval['char_id_list'].apply(len)

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=882,882,15,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 15, 15, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 15, 9, 1, 0]"
1,(6+1)*5=35,35,11,"[2, 13, 5, 8, 3, 4, 12, 17, 10, 12, 1]","[0, 0, 0, 0, 0, 0, 0, 10, 12, 1, 0]"
2,13+32+29=74,74,12,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 14, 11, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 1, 0]"
3,31*(3-11)=-248,-248,15,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1, 0]"
4,24*49+1=1177,1177,13,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 14, 14, 1]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 14, 14, 1, 0]"


In [13]:
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.01
grad_clip = 1

In [14]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return len(self.sequences)

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        # x 是輸入序列的 ID 列表 (char_id_list)
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here
        # y 是目標序列的 ID 列表 (label_id_list)
        y = self.sequences.iloc[index]['label_id_list'] # Write your code here
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [15]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train,
    batch_size=batch_size,
    shuffle=True,       # 在訓練時打亂資料順序，這點非常重要
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

ds_eval = Dataset(df_eval[['char_id_list', 'label_id_list']])

dl_eval = torch.utils.data.DataLoader(
    dataset=ds_eval,
    batch_size=batch_size,
    shuffle=False,       # 驗證階段不要打亂
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [16]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
        
            char_list = [char_to_id[c] for c in start_char]
        
            next_char = None

            # 將模型設置為評估模式，這會關閉 Dropout 等層
            self.eval()

            # 使用 torch.no_grad() 上下文管理器，在此區塊中不計算梯度，以加速並節省記憶體
            with torch.no_grad():
                while len(char_list) < max_len: 
                    # --- 填入的程式碼開始 ---

                    # 1. 準備模型的輸入張量
                    #    - 需要與模型在同一個設備上 (例如 'cuda' 或 'cpu')
                    device = next(self.parameters()).device
                    #    - 輸入需要有批次維度，所以我們將 char_list 包在另一個列表中
                    #    - 維度變為: [1, 當前序列長度]
                    input_tensor = torch.tensor([char_list], dtype=torch.long).to(device)
                
                    # 模型的 forward 方法也需要序列的實際長度
                    input_length = torch.tensor([len(char_list)], dtype=torch.long)
                
                    # 2. 將輸入傳入模型以獲得預測的 logits
                    #    - 輸出 y 的維度將是: [批次大小, 序列長度, 字典大小]
                    #    - 在此情境下是: [1, len(char_list), vocab_size]
                    y = self.forward(input_tensor, input_length)
                
                    # 3. 我們只關心對「下一個」字元的預測，這對應於序列中「最後一個」時間點的輸出
                    #    - 我們使用 y[:, -1, :] 來選取這個部分，得到維度為 [1, vocab_size] 的張量
                    last_time_step_logits = y[:, -1, :]
                
                    # 4. 使用 argmax 找出分數最高的那個字元的 ID
                    #    - torch.argmax 回傳的是一個張量，所以使用 .item() 來提取純粹的 Python 數字
                    next_char = torch.argmax(last_time_step_logits, dim=1).item()
                
                    # --- 填入的程式碼結束 ---

                    # 5. 檢查生成的字元是否為序列結束符號
                    if next_char == char_to_id['<eos>']:
                        break
                
                    # 6. 如果不是，則將新字元的 ID 加入到我們的列表中，並繼續迴圈
                    char_list.append(next_char)
            
            # 將最終的 ID 列表轉換回字元，並回傳結果
            return [id_to_char[ch_id] for ch_id in char_list]

In [17]:
torch.manual_seed(2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim).to(device) # 將模型移動到指定的設備上

In [18]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# 使用 Adam 優化器
# optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) 

In [19]:
from tqdm import tqdm
from copy import deepcopy
import torch

# 假設 model, device, epochs, dl_train, optimizer, criterion,
# vocab_size, grad_clip, df_eval, char_to_id, id_to_char 都已定義

# 將模型設為訓練模式 (啟用 Dropout 等)
model = model.to(device)
model.train()

# i 用於控制何時印出 loss
i = 0
for epoch in range(1, epochs+1):
    # --- 訓練迴圈 ---
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        
        # --- 填空 1: 清除梯度 ---
        # 在計算新的梯度前，必須先清除上一步遺留的梯度
        optimizer.zero_grad()
    
        # 將資料移動到 GPU/CPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 模型前向傳播，得到預測結果
        # batch_pred_y 的維度: [batch_size, seq_len, vocab_size]
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # --- 填空 2: 計算損失 & 反向傳播 ---
        # CrossEntropyLoss 要求 pred 維度為 (N, C) 和 target 維度為 (N)
        # 所以我們需要將 batch 維度和 seq_len 維度攤平
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = batch_y.view(-1)
        
        loss = criterion(pred_view, target_view)
        
        # 根據損失計算梯度
        loss.backward()

        # --- ✨ 核心修改：移除梯度裁剪 ✨ ---
        # 在這個實驗版本中，我們註解掉這一行，讓梯度自由流動
        # torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) 

        # --- 填空 3: 更新模型參數 ---
        # 優化器根據「未經裁剪」的梯度來更新模型的權重
        optimizer.step()

        i += 1
        if i % 50 == 0:
            # 如果 loss 變成 nan，訓練可能已經失敗，提前印出訊息
            if torch.isnan(loss):
                print("\n💥 Loss has become NaN. Training failed due to unstable gradients.")
                break
            bar.set_postfix(loss=loss.item())
    
    # 如果訓練失敗，提前跳出 epoch 迴圈
    if torch.isnan(loss):
        break

    # --- 評估迴圈 ---
    # 將模型切換到評估模式 (關閉 Dropout 等)
    model.eval()
    
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        # 從 DataFrame 中取出問題和標準答案
        # batch_x 是字串，例如 '10+4='
        # batch_y 也是字串，例如 '14'
        batch_x = row['src']
        batch_y = str(row['tgt'])
        
        # --- 填空 4: 使用 generator 生成預測 ---
        prediction = model.generator(batch_x)
        
        # 從生成結果中提取答案部分
        try:
            equal_idx = prediction.index('=')
            predicted_answer = "".join(prediction[equal_idx+1:])
        except ValueError:
            predicted_answer = ""
            
        # --- 填空 5: 比較預測與標準答案 ---
        if predicted_answer == batch_y:
            matched += 1
        
        total += 1
        
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # --- 將模型切換回訓練模式，為下一個 epoch 做準備 ---
    model.train()
        
    print(f"\nEpoch {epoch} Validation Accuracy: {matched/total:.4f}")


Train epoch 1: 100%|██████████| 37020/37020 [02:45<00:00, 223.39it/s, loss=0.573]
Validation epoch 1: 263250it [41:57, 104.56it/s, accuracy=0.3417]



Epoch 1 Validation Accuracy: 0.3417


Train epoch 2: 100%|██████████| 37020/37020 [02:45<00:00, 223.47it/s, loss=0.555]
Validation epoch 2: 263250it [39:26, 111.26it/s, accuracy=0.3608]



Epoch 2 Validation Accuracy: 0.3608


Train epoch 3: 100%|██████████| 37020/37020 [02:45<00:00, 223.66it/s, loss=0.528]
Validation epoch 3: 263250it [39:13, 111.86it/s, accuracy=0.4276]


...


Epoch 8 Validation Accuracy: 0.4214


Train epoch 9: 100%|██████████| 37020/37020 [02:45<00:00, 223.05it/s, loss=0.554]
Validation epoch 9: 263250it [39:13, 111.84it/s, accuracy=0.4280]



Epoch 9 Validation Accuracy: 0.4280


Train epoch 10: 100%|██████████| 37020/37020 [02:45<00:00, 223.35it/s, loss=0.434]
Validation epoch 10: 263250it [40:09, 109.25it/s, accuracy=0.4678]



Epoch 10 Validation Accuracy: 0.4678


# E.	Seq2Seq Encoder–Decoder + input reversal

In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
# === 資料預處理：Encoder-Decoder Optimization ===
# 為了提升模型對長序列的記憶與處理能力，我在此階段調整了資料流向，並引入翻轉序列技術。

# 1. 標籤標準化
# 將目標變數 (tgt) 統一轉換為字串格式，以確保後續在建構詞彙表與計算損失時的型別一致性。
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))


# 2. 核心優化：輸入序列翻轉 
# 參考神經機器翻譯（NMT）的經典文獻，我選擇將來源序列 (src) 進行翻轉。
df_train['src_reversed'] = df_train['src'].apply(lambda x: x[::-1])
df_eval['src_reversed'] = df_eval['src'].apply(lambda x: x[::-1])


# 3. 序列特徵統計
# 為了後續動態填充 與 Bucket 排序的效率考量，我分別計算並記錄了來源與目標序列的長度。
df_train['src_len'] = df_train['src_reversed'].apply(len)
df_train['tgt_len'] = df_train['tgt'].apply(len)
df_eval['src_len'] = df_eval['src_reversed'].apply(len)
df_eval['tgt_len'] = df_eval['tgt'].apply(len)


In [4]:
# === 構建字元映射表 (Dictionary Building) ===
char_to_id = {}
id_to_char = {}

# 加特殊符號
char_to_id['<pad>'] = 0  # 用於填充序列
char_to_id['<eos>'] = 1  # 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集資料集中的唯一字元
# 彙整訓練資料中出現過的所有字元，為建立詞彙表做準備。
unique_chars = set(''.join(df_train['src']))
# 移除已手動定義的特殊符號，避免重複分配 ID
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# 穩定性優化：排序字元
# 透過排序確保每次執行程式時，字元與 ID 的對應關係保持一致，避免模型訓練的隨機性。
sorted_chars = sorted(list(unique_chars))

# 每個字符分配ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))

Vocab size: 18


In [5]:
# === 資料處理流程優化：編碼器-解碼器 (Encoder-Decoder) 結構封裝 ===
# 為了適應 Seq2Seq 架構，我重新設計了資料 pipeline，將輸入與輸出拆解為三個核心部分。

# 1. 構建編碼器輸入
# 我使用了先前翻轉後的 'src_reversed' 欄位。
# 透過翻轉算式，我能縮短運算元與答案開頭的距離，優化編碼器的特徵提取效果。
df_train['encoder_input_ids'] = df_train['src_reversed'].apply(
    lambda x: [char_to_id[c] for c in x]
)
df_eval['encoder_input_ids'] = df_eval['src_reversed'].apply(
    lambda x: [char_to_id[c] for c in x]
)


# 2. 構建解碼器目標標籤 
# 這是模型訓練時的「標準答案」。
# 我在序列末尾特別附加了 <eos> 標記，這是我用來引導解碼器學習「何時該停止生成」的關鍵信號。
df_train['decoder_label_ids'] = df_train['tgt'].apply(
    lambda x: [char_to_id[c] for c in x] + [char_to_id['<eos>']]
)
df_eval['decoder_label_ids'] = df_eval['tgt'].apply(
    lambda x: [char_to_id[c] for c in x] + [char_to_id['<eos>']]
)


# 3. 構建解碼器輸入 
# 將標籤序列向右平移，並在開頭補上 <eos> 作為起始符號 (SOS)。
# 這樣模型在預測第 t 個字元時，能參考第 t-1 個字元的真實 ID，從而加速收斂。
df_train['decoder_input_ids'] = df_train['decoder_label_ids'].apply(lambda x: [char_to_id['<eos>']] + x[:-1])
df_eval['decoder_input_ids'] = df_eval['decoder_label_ids'].apply(lambda x: [char_to_id['<eos>']] + x[:-1])

# 4. 長度特徵統計 
df_train['encoder_len'] = df_train['encoder_input_ids'].apply(len)
df_train['decoder_len'] = df_train['decoder_label_ids'].apply(len)

df_eval['encoder_len'] = df_eval['encoder_input_ids'].apply(len)
df_eval['decoder_len'] = df_eval['decoder_label_ids'].apply(len)

# 顯示 DataFrame 
print("處理完成後的新 DataFrame 結構：")
df_train.head()

處理完成後的新 DataFrame 結構：


,src,tgt,src_reversed,src_len,tgt_len,encoder_input_ids,decoder_label_ids,decoder_input_ids,encoder_len,decoder_len
0,14*(43+20)=,882,=)02+34(*41,11,3,"[17, 3, 7, 9, 5, 10, 11, 2, 4, 11, 8]","[15, 15, 9, 1]","[1, 15, 15, 9]",11,4
1,(6+1)*5=,35,=5*)1+6(,8,2,"[17, 12, 4, 3, 8, 5, 13, 2]","[10, 12, 1]","[1, 10, 12]",8,3
2,13+32+29=,74,=92+23+31,9,2,"[17, 16, 9, 5, 9, 10, 5, 10, 8]","[14, 11, 1]","[1, 14, 11]",9,3
3,31*(3-11)=,-248,=)11-3(*13,10,4,"[17, 3, 8, 8, 6, 10, 2, 4, 8, 10]","[6, 9, 11, 15, 1]","[1, 6, 9, 11, 15]",10,5
4,24*49+1=,1177,=1+94*42,8,4,"[17, 8, 5, 16, 11, 4, 11, 9]","[8, 8, 14, 14, 1]","[1, 8, 8, 14, 14]",8,5


In [6]:
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001 
grad_clip = 1

In [7]:
import torch

class Dataset(torch.utils.data.Dataset):
    """
    自定義數據集類別：針對 Encoder-Decoder 架構封裝訓練樣本。
    我將資料解構為三個核心序列，以精確支援 Seq2Seq 的訓練機制。
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # 回傳資料集總量，供 DataLoader 進行取樣控制
        return len(self.sequences)

    def __getitem__(self, index):
        # 從 DataFrame 中提取特定索引的預處理 ID 序列
        # 編碼器輸入 
        encoder_x = self.sequences.iloc[index]['encoder_input_ids']
        
        # 解碼器輸入
        decoder_x = self.sequences.iloc[index]['decoder_input_ids']

        # 計算 Loss
        # 模型必須學習生成此序列，並在結尾有<eos> 標記。
        decoder_y = self.sequences.iloc[index]['decoder_label_ids']
        
        return encoder_x, decoder_x, decoder_y

In [8]:
# === Collate function：動態序列填充與對齊 ===

def collate_fn(batch):
    # 將序列轉換為 PyTorch Tensor，為了進入GPU準備。
    batch_encoder_x = [torch.tensor(data[0]) for data in batch]
    batch_decoder_x = [torch.tensor(data[1]) for data in batch]
    batch_decoder_y = [torch.tensor(data[2]) for data in batch]

    # 2. 序列長度統計
    # 記錄編碼器與解碼器的原始長度。這對 RNN 處理變長序列至關重要，
    # 讓我能在後續使用 pack_padded_sequence 來優化計算效能，避開無意義的填充運算。
    batch_encoder_lens = torch.LongTensor([len(x) for x in batch_encoder_x])
    # 由於解碼器的輸入 (Teacher Forcing) 與目標 (Labels) 為平移關係，長度一致，故統計一次即可。
    batch_decoder_lens = torch.LongTensor([len(y) for y in batch_decoder_y])

    # 3. 動態填充 
    
    # (1) 編碼器輸入填充 
    pad_batch_encoder_x = torch.nn.utils.rnn.pad_sequence(
        batch_encoder_x,
        batch_first=True,
        padding_value=char_to_id['<pad>']
    )

    # (2) 解碼器輸入填充
    pad_batch_decoder_x = torch.nn.utils.rnn.pad_sequence(
        batch_decoder_x,
        batch_first=True,
        padding_value=char_to_id['<pad>']
    )

    # (3) 解碼器目標填充 
    pad_batch_decoder_y = torch.nn.utils.rnn.pad_sequence(
        batch_decoder_y,
        batch_first=True,
        padding_value=char_to_id['<pad>']
    )

    # 回傳張量與長度資訊
    return (pad_batch_encoder_x, pad_batch_decoder_x, pad_batch_decoder_y,
            batch_encoder_lens, batch_decoder_lens)

In [9]:
# 建立訓練集的 Dataset 和 DataLoader
ds_train = Dataset(df_train)
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train,
    batch_size=batch_size,
    shuffle=True,       # 在訓練時打亂資料順序
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# 建立驗證集的 Dataset 和 DataLoader 
ds_eval = Dataset(df_eval)
dl_eval = torch.utils.data.DataLoader(
    dataset=ds_eval,
    batch_size=batch_size,
    shuffle=False,     
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print("DataLoader 準備完成！")

DataLoader 準備完成！


In [10]:
import torch

# ======================================================================
#  1. 定義 Encoder: 讀取輸入序列，並將其壓縮成一個 "上下文向量"
# ======================================================================
class EncoderRNN(torch.nn.Module):
    def __init__(self, embedding_layer, hidden_dim):
        super(EncoderRNN, self).__init__()
        self.embedding = embedding_layer
        
        # 兩層 LSTM
        self.rnn_layer1 = torch.nn.LSTM(
            input_size=self.embedding.embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.rnn_layer2 = torch.nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

    def forward(self, batch_x, batch_x_lens):
        # Embedding
        embedded_x = self.embedding(batch_x)
        
        # 打包變長序列
        packed_x = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_x,
            batch_x_lens.cpu(), 
            batch_first=True,
            enforce_sorted=False
        )
        
        # 通過 LSTM
        _, hidden_state_l1 = self.rnn_layer1(packed_x)
        _, hidden_state_l2 = self.rnn_layer2(packed_x, hidden_state_l1)
        
        # 返回最後一層的 hidden 和 cell state 作為上下文
        return hidden_state_l2

# ======================================================================
#  解碼器 (Decoder)：負責「生成」最終答案
# 它接收來自 Encoder 的上下文向量，並在 Teacher Forcing 的引導下逐字產出目標序列。
# ======================================================================
class DecoderRNN(torch.nn.Module):
    def __init__(self, embedding_layer, hidden_dim, vocab_size):
        super(DecoderRNN, self).__init__()
        self.embedding = embedding_layer
        
        # 兩層 LSTM
        self.rnn_layer1 = torch.nn.LSTM(
            input_size=self.embedding.embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.rnn_layer2 = torch.nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=hidden_dim, out_features=vocab_size)
        )

    def forward(self, batch_x, initial_hidden_state):
        # Embedding
        embedded_x = self.embedding(batch_x)
        
        # 通過 LSTM，使用來自 Encoder hidden state 作為初始狀態
        outputs, _ = self.rnn_layer1(embedded_x, initial_hidden_state)
        outputs, _ = self.rnn_layer2(outputs) 
        
        # 得到預測分數
        logits = self.linear(outputs)
        
        return logits

# ======================================================================
#  Seq2Seq 主模型：整合編解碼流程
# ======================================================================
class Seq2Seq(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Seq2Seq, self).__init__()
        
        # 建立一個共享的 Embedding 層
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=char_to_id['<pad>']
        )
        
        # Encoder 和 Decoder
        self.encoder = EncoderRNN(self.embedding, hidden_dim)
        self.decoder = DecoderRNN(self.embedding, hidden_dim, vocab_size)

    def forward(self, encoder_x, decoder_x, encoder_lens):
        # 將輸入序列傳給 Encoder，得到上下文向量 
        encoder_hidden = self.encoder(encoder_x, encoder_lens)
        
        # 解碼器輸入和 Encoder 的上下文向量傳給 Decoder
        logits = self.decoder(decoder_x, encoder_hidden)
        
        return logits

    # ======================================================================
    #  生成器
    # ======================================================================
    def generator(self, src_text, max_len=50): # max_len 現在是答案的最大長度
        self.eval()
        device = next(self.parameters()).device

        # 將輸入的數學式翻轉並轉換為張量
        reversed_src_text = src_text[::-1]
        encoder_x = torch.tensor([[char_to_id[c] for c in reversed_src_text]], dtype=torch.long).to(device)
        encoder_lens = torch.tensor([len(reversed_src_text)], dtype=torch.long)
        
        with torch.no_grad():
            # 取得 Encoder 的上下文向量
            encoder_hidden = self.encoder(encoder_x, encoder_lens)

        # 解碼器的初始輸入是 <eos> (作為起始符號 <sos>)
        decoder_input = torch.tensor([[char_to_id['<eos>']]], dtype=torch.long).to(device)
        
        # 將 Encoder 的輸出作為 Decoder 的初始隱藏狀態
        decoder_hidden = encoder_hidden
        
        output_ids = []
        
        for _ in range(max_len):
            with torch.no_grad():
                # 將當前輸入和隱藏狀態傳給 Decoder
                logits = self.decoder(decoder_input, decoder_hidden)
                
            # 找出分數最高的預測
            top1_id = logits.argmax(dim=-1).item()

            # 如果預測出 <eos>，表示結束
            if top1_id == char_to_id['<eos>']:
                break
            
            output_ids.append(top1_id)
            
            # 將當前預測的結果作為下一步的輸入
            decoder_input = torch.tensor([[top1_id]], dtype=torch.long).to(device)
            
        return "".join([id_to_char[id] for id in output_ids])

In [11]:
import torch
import torch.optim as optim

torch.manual_seed(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Seq2Seq 模型
model = Seq2Seq(vocab_size,
                embed_dim,
                hidden_dim).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

print("模型、損失函數和優化器均已準備就緒！")
print(model) # 模型結構

模型、損失函數和優化器均已準備就緒！
Seq2Seq(
  (embedding): Embedding(18, 256, padding_idx=0)
  (encoder): EncoderRNN(
    (embedding): Embedding(18, 256, padding_idx=0)
    (rnn_layer1): LSTM(256, 256, batch_first=True)
    (rnn_layer2): LSTM(256, 256, batch_first=True)
  )
  (decoder): DecoderRNN(
    (embedding): Embedding(18, 256, padding_idx=0)
    (rnn_layer1): LSTM(256, 256, batch_first=True)
    (rnn_layer2): LSTM(256, 256, batch_first=True)
    (linear): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=18, bias=True)
    )
  )
)


In [12]:
from tqdm import tqdm
from copy import deepcopy
import torch

model = model.to(device)

# 紀錄最佳的模型
best_accuracy = 0.0
best_model_state = None

# i 控制何時印出 loss
i = 0
for epoch in range(1, epochs + 1):

    # 訓練迴圈 
    model.train() 
    
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch in bar:
        (encoder_x, decoder_x, decoder_y, 
         encoder_lens, decoder_lens) = [item.to(device) for item in batch]
        
        # 清除梯度
        optimizer.zero_grad()
        
        # 模型前向傳播 
        # 呼叫 Seq2Seq 模型需要傳入 encoder_x, decoder_x 和 encoder_lens
        batch_pred_y = model(encoder_x, decoder_x, encoder_lens)
        
        #計算損失 & 反向傳播
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = decoder_y.view(-1)
        loss = criterion(pred_view, target_view)
        loss.backward()

        # 梯度裁剪 
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip) 

        # 更新模型參數
        optimizer.step()
        
        i += 1
        if i % 50 == 0:
            bar.set_postfix(loss=loss.item())
            
    #評估迴圈
    model.eval() 
    
    matched = 0
    total = 0
    # 直接迭代 DataFrame
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}", total=len(df_eval))
    for _, row in bar_eval:
        # 取出原始問題和標準答案字串
        src_text = row['src']
        target_text = str(row['tgt'])
        
        # 使用 generator 生成預測 
        # 直接傳入原始 src_text
        predicted_answer = model.generator(src_text)
        
        # 比較預測與標準答案
        if predicted_answer == target_text:
            matched += 1
        
        total += 1
        
        # 更新
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # 計算當前 epoch 的準確率
    current_accuracy = matched / total if total > 0 else 0
    
    #儲存最佳模型 
    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_model_state = deepcopy(model.state_dict())
        print(f"\n✨ New best accuracy: {best_accuracy:.4f}! Saving model...")

    print(f"Epoch {epoch} Validation Accuracy: {current_accuracy:.4f}")

# 訓練結束後，載入表現最好的模型
if best_model_state:
    model.load_state_dict(best_model_state)
    print(f"\nTraining finished. Loaded best model with accuracy: {best_accuracy:.4f}")

Validation epoch 1: 100%|██████████| 263250/263250 [2:18:49<00:00, 31.61it/s, accuracy=0.0781]  



✨ New best accuracy: 0.0781! Saving model...
Epoch 1 Validation Accuracy: 0.0781


Validation epoch 2: 100%|██████████| 263250/263250 [2:10:33<00:00, 33.61it/s, accuracy=0.0810]  



✨ New best accuracy: 0.0810! Saving model...
Epoch 2 Validation Accuracy: 0.0810


Validation epoch 3: 100%|██████████| 263250/263250 [2:09:39<00:00, 33.84it/s, accuracy=0.0819]  


...


✨ New best accuracy: 0.1106! Saving model...
Epoch 8 Validation Accuracy: 0.1106


Validation epoch 9: 100%|██████████| 263250/263250 [2:05:37<00:00, 34.93it/s, accuracy=0.1069]  


Epoch 9 Validation Accuracy: 0.1069


Validation epoch 10: 100%|██████████| 263250/263250 [2:09:46<00:00, 33.81it/s, accuracy=0.1157]  



✨ New best accuracy: 0.1157! Saving model...
Epoch 10 Validation Accuracy: 0.1157

Training finished. Loaded best model with accuracy: 0.1157
